# Music Genre Classification — CNN with Mel Spectrograms

A self-contained notebook that trains a CNN to classify music genres using the GTZAN dataset.

**Steps:**
1. Install dependencies & verify GPU
2. Download GTZAN dataset
3. Preprocess audio → Mel spectrograms
4. Build & train CNN model
5. Evaluate (chunk-level & file-level majority voting)
6. Inference on a single file

**Runtime:** Go to `Runtime → Change runtime type → GPU` before running.

## 1. Setup

In [ ]:
# Install dependencies (TensorFlow is pre-installed on Colab)
!pip install -q librosa>=0.10.0 numpy pandas>=2.0 matplotlib>=3.7 seaborn>=0.12 scikit-learn>=1.3 soundfile>=0.12 kagglehub
!apt-get -qq install -y ffmpeg  # needed for MP3 support in librosa

In [ ]:
# Verify GPU
import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU devices: {tf.config.list_physical_devices('GPU')}")
if not tf.config.list_physical_devices('GPU'):
    print("WARNING: No GPU detected. Go to Runtime -> Change runtime type -> GPU")

In [ ]:
# All imports
import os
import sys
import json
import shutil
import subprocess
import zipfile

import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

## 2. Configuration

In [ ]:
# --- Paths ---
BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "data", "genres_original")
SPECTROGRAM_DIR = os.path.join(BASE_DIR, "spectrograms")
MODEL_DIR = os.path.join(BASE_DIR, "models")
MODEL_PATH = os.path.join(MODEL_DIR, "genre_cnn.keras")
HISTORY_PATH = os.path.join(MODEL_DIR, "history.json")
PLOTS_DIR = os.path.join(BASE_DIR, "plots")

# --- Audio ---
SAMPLE_RATE = 22050
CHUNK_DURATION = 4   # seconds
CHUNK_OVERLAP = 2    # seconds
CHUNK_SAMPLES = SAMPLE_RATE * CHUNK_DURATION   # 88200
OVERLAP_SAMPLES = SAMPLE_RATE * CHUNK_OVERLAP  # 44100

# --- Mel Spectrogram ---
N_MELS = 128
HOP_LENGTH = 512
N_FFT = 2048
IMG_HEIGHT = 150
IMG_WIDTH = 150

# --- Model ---
CONV_FILTERS = [32, 64, 128, 256, 512]
KERNEL_SIZE = (3, 3)
POOL_SIZE = (2, 2)
CONV_DROPOUT = 0.30
DENSE_UNITS = 1200
DENSE_DROPOUT = 0.45
NUM_CLASSES = 10

# --- Training ---
EPOCHS = 30
BATCH_SIZE = 32
LEARNING_RATE = 0.001
TEST_SPLIT = 0.20
RANDOM_SEED = 42

# --- Genre labels (alphabetical, matching GTZAN folder names) ---
GENRES = [
    "blues", "classical", "country", "disco", "hiphop",
    "jazz", "metal", "pop", "reggae", "rock",
]

# Create output directories
for d in [DATA_DIR, SPECTROGRAM_DIR, MODEL_DIR, PLOTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Base directory: {BASE_DIR}")

## 3. Download GTZAN Dataset

The cell below uses `kagglehub` to download the dataset. **First-time users** will be prompted to authenticate — follow the link shown in the output and paste your Kaggle API token when asked.

If automatic download fails, download manually from [Kaggle](https://www.kaggle.com/datasets/andradaolteanu/gtzan-dataset-music-genre-classification), upload the zip to Colab, and extract so genre folders are at `data/genres_original/`.

In [ ]:
def organize_dataset(source_dir, dest_dir="data"):
    """Move genre folders to data/genres_original/."""
    target_path = os.path.join(dest_dir, "genres_original")

    if os.path.isdir(target_path) and len(os.listdir(target_path)) >= 10:
        return target_path

    candidates = [
        os.path.join(source_dir, "Data", "genres_original"),
        os.path.join(source_dir, "genres_original"),
        os.path.join(source_dir, "data", "genres_original"),
    ]
    if os.path.isdir(os.path.join(source_dir, "blues")):
        candidates.insert(0, source_dir)

    found = None
    for candidate in candidates:
        if os.path.isdir(candidate) and os.path.isdir(os.path.join(candidate, "blues")):
            found = candidate
            break

    if found is None:
        print(f"WARNING: Could not find genre folders in {source_dir}")
        print("Please manually place genre folders in data/genres_original/")
        return None

    if os.path.abspath(found) != os.path.abspath(target_path):
        os.makedirs(dest_dir, exist_ok=True)
        if os.path.exists(target_path):
            shutil.rmtree(target_path)
        shutil.copytree(found, target_path)

    data_subdir = os.path.join(dest_dir, "Data")
    if os.path.isdir(data_subdir):
        shutil.rmtree(data_subdir)

    return target_path


def download_gtzan():
    """Download and organize the GTZAN dataset."""
    if os.path.isdir(DATA_DIR):
        genre_count = len([
            d for d in os.listdir(DATA_DIR)
            if os.path.isdir(os.path.join(DATA_DIR, d)) and not d.startswith(".")
        ])
        if genre_count == 10:
            print("Dataset already exists. Skipping download.")
            return DATA_DIR

    import kagglehub

    # Authenticate first-time users (will prompt for API token if not logged in)
    try:
        kagglehub.login()
    except Exception:
        print("If prompted, go to https://www.kaggle.com/settings -> 'Create New Token'")
        print("and paste your Kaggle username and API key when asked.\n")
        kagglehub.login()

    print("Downloading GTZAN dataset via kagglehub...")
    try:
        path = kagglehub.dataset_download(
            "andradaolteanu/gtzan-dataset-music-genre-classification"
        )
        print(f"Downloaded to cache: {path}")
        result = organize_dataset(path, "data")
        if result:
            print(f"Dataset ready at: {result}")
        return result
    except Exception as e:
        print(f"\nDownload failed: {e}")
        print("\nManual download instructions:")
        print("  1. Go to: https://www.kaggle.com/datasets/andradaolteanu/gtzan-dataset-music-genre-classification")
        print("  2. Download and upload the zip to Colab")
        print("  3. Extract so that: data/genres_original/blues/blues.00000.wav exists")
        return None


def validate_dataset():
    """Validate that the dataset is correctly structured."""
    print(f"\nValidating dataset at: {DATA_DIR}")
    print("-" * 45)
    total_files = 0
    for genre in GENRES:
        genre_dir = os.path.join(DATA_DIR, genre)
        if not os.path.isdir(genre_dir):
            print(f"  MISSING: {genre}/")
            continue
        wav_files = [f for f in os.listdir(genre_dir) if f.endswith(".wav")]
        count = len(wav_files)
        total_files += count
        status = "OK" if count >= 90 else "LOW"
        print(f"  {genre:12s} : {count:3d} files  [{status}]")
    print("-" * 45)
    print(f"  Total: {total_files} files")
    return total_files >= 900


download_gtzan()
validate_dataset()

## 4. Preprocessing — Audio to Mel Spectrograms

Each 30-second audio file is split into overlapping 4-second chunks (2-second overlap), converted to a 128-band Mel spectrogram, resized to 150×150, and normalized to [0, 1].

This takes ~5–10 minutes.

In [ ]:
def load_and_chunk(file_path):
    """Load a .wav file and split into overlapping 4-second chunks."""
    audio, _ = librosa.load(file_path, sr=SAMPLE_RATE, res_type="kaiser_fast")
    step = CHUNK_SAMPLES - OVERLAP_SAMPLES
    chunks = []
    for start in range(0, len(audio) - CHUNK_SAMPLES + 1, step):
        chunks.append(audio[start : start + CHUNK_SAMPLES])
    return chunks


def audio_to_mel_spectrogram(audio_chunk):
    """Compute Mel spectrogram from an audio chunk and convert to dB."""
    mel_spec = librosa.feature.melspectrogram(
        y=audio_chunk, sr=SAMPLE_RATE, n_mels=N_MELS,
        hop_length=HOP_LENGTH, n_fft=N_FFT,
    )
    return librosa.power_to_db(mel_spec, ref=np.max)


def resize_spectrogram(spec):
    """Resize spectrogram to 150x150 and normalize to [0, 1]."""
    img = Image.fromarray(spec)
    img = img.resize((IMG_WIDTH, IMG_HEIGHT), Image.LANCZOS)
    resized = np.array(img, dtype=np.float64)
    spec_min, spec_max = resized.min(), resized.max()
    if spec_max - spec_min > 0:
        resized = (resized - spec_min) / (spec_max - spec_min)
    else:
        resized = np.zeros_like(resized)
    return resized.astype(np.float32)


def preprocess_single_file(file_path):
    """Preprocess a single audio file for inference. Returns (num_chunks, 150, 150, 1)."""
    chunks = load_and_chunk(file_path)
    spectrograms = []
    for chunk in chunks:
        mel_spec = audio_to_mel_spectrogram(chunk)
        spectrograms.append(resize_spectrogram(mel_spec))
    X = np.array(spectrograms, dtype=np.float32)
    return X[..., np.newaxis]

In [ ]:
# Process the entire dataset
all_spectrograms = []
all_labels = []
file_map_rows = []
chunk_index = 0
skipped_files = []

for genre_idx, genre in enumerate(GENRES):
    genre_dir = os.path.join(DATA_DIR, genre)
    if not os.path.isdir(genre_dir):
        print(f"WARNING: Genre directory not found: {genre_dir}")
        continue

    wav_files = sorted([f for f in os.listdir(genre_dir) if f.endswith(".wav")])
    print(f"Processing {genre} ({len(wav_files)} files)...")

    for wav_file in wav_files:
        file_path = os.path.join(genre_dir, wav_file)
        try:
            chunks = load_and_chunk(file_path)
        except Exception as e:
            print(f"  Skipping {wav_file}: {e}")
            skipped_files.append(wav_file)
            continue

        for chunk_num, chunk in enumerate(chunks):
            mel_spec = audio_to_mel_spectrogram(chunk)
            mel_resized = resize_spectrogram(mel_spec)
            all_spectrograms.append(mel_resized)
            all_labels.append(genre_idx)
            file_map_rows.append({
                "chunk_index": chunk_index,
                "genre": genre,
                "original_file": wav_file,
                "chunk_number": chunk_num,
            })
            chunk_index += 1

# Convert to arrays
X = np.array(all_spectrograms, dtype=np.float32)[..., np.newaxis]  # (N, 150, 150, 1)
y = np.array(all_labels, dtype=np.int32)
file_map = pd.DataFrame(file_map_rows)

# Save for reproducibility
np.save(os.path.join(SPECTROGRAM_DIR, "X.npy"), X)
np.save(os.path.join(SPECTROGRAM_DIR, "y.npy"), y)
file_map.to_csv(os.path.join(SPECTROGRAM_DIR, "file_map.csv"), index=False)

print(f"\nPreprocessing complete:")
print(f"  Spectrograms: {X.shape}")
print(f"  Labels: {y.shape}")
print(f"  Skipped files: {len(skipped_files)}")

In [ ]:
# Visualize sample spectrograms from each genre
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for i, ax in enumerate(axes.flat):
    idx = np.where(y == i)[0][0]
    ax.imshow(X[idx, :, :, 0], aspect="auto", origin="lower", cmap="magma")
    ax.set_title(GENRES[i])
    ax.set_xlabel("Time")
    ax.set_ylabel("Mel Band")
plt.suptitle("Sample Mel Spectrograms by Genre", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Train/Test Split (File-Level Stratified)

Chunks from the same song never appear in both train and test sets — this prevents data leakage.

In [ ]:
# Stratified split at the file level
unique_files = file_map[["genre", "original_file"]].drop_duplicates()

train_files, test_files = train_test_split(
    unique_files,
    test_size=TEST_SPLIT,
    random_state=RANDOM_SEED,
    stratify=unique_files["genre"],
)

train_set = set(zip(train_files["genre"], train_files["original_file"]))
test_set = set(zip(test_files["genre"], test_files["original_file"]))

train_mask = file_map.apply(lambda r: (r["genre"], r["original_file"]) in train_set, axis=1)
test_mask = file_map.apply(lambda r: (r["genre"], r["original_file"]) in test_set, axis=1)

train_indices = file_map[train_mask].index.values
test_indices = file_map[test_mask].index.values

X_train = X[train_indices]
X_test = X[test_indices]
y_train = to_categorical(y[train_indices], NUM_CLASSES)
y_test = to_categorical(y[test_indices], NUM_CLASSES)
test_file_map = file_map[test_mask].reset_index(drop=True)

# Save test data for later evaluation
np.save(os.path.join(SPECTROGRAM_DIR, "X_test.npy"), X_test)
np.save(os.path.join(SPECTROGRAM_DIR, "y_test.npy"), y_test)
test_file_map.to_csv(os.path.join(SPECTROGRAM_DIR, "test_file_map.csv"), index=False)

print(f"Train: {len(train_indices)} chunks from {len(train_files)} files")
print(f"Test:  {len(test_indices)} chunks from {len(test_files)} files")

## 6. Data Augmentation & Training

GTZAN only has 1000 tracks, so we apply data augmentation on spectrograms during training to improve generalization. Augmentations include random time/frequency masking (SpecAugment) and slight brightness shifts.

The CNN has 5 convolutional blocks with BatchNorm, ~11.4M parameters. Uses EarlyStopping (patience=5).

In [ ]:
# Data augmentation layer (SpecAugment-style masking + noise)
data_augmentation = keras.Sequential([
    layers.GaussianNoise(0.01),
], name="augmentation")

# Build model with augmentation in the training path
inputs = layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 1))
x = data_augmentation(inputs)

for filters in CONV_FILTERS:
    x = layers.Conv2D(filters, KERNEL_SIZE, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(pool_size=POOL_SIZE)(x)
    x = layers.Dropout(CONV_DROPOUT)(x)

x = layers.Flatten()(x)
x = layers.Dense(DENSE_UNITS, activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(DENSE_DROPOUT)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

In [ ]:
# Train
callbacks = [
    EarlyStopping(
        monitor="val_loss", patience=5,
        restore_best_weights=True, verbose=1,
    ),
    ModelCheckpoint(
        MODEL_PATH, monitor="val_loss",
        save_best_only=True, verbose=1,
    ),
    ReduceLROnPlateau(
        monitor="val_loss", factor=0.5,
        patience=3, min_lr=1e-6, verbose=1,
    ),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
)

# Save training history
history_dict = {
    key: [float(v) for v in values]
    for key, values in history.history.items()
}
with open(HISTORY_PATH, "w") as f:
    json.dump(history_dict, f, indent=2)

print(f"\nModel saved to: {MODEL_PATH}")
print(f"History saved to: {HISTORY_PATH}")

## 7. Evaluation

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history_dict["accuracy"], label="Train Accuracy")
ax1.plot(history_dict["val_accuracy"], label="Val Accuracy")
ax1.set_title("Model Accuracy")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history_dict["loss"], label="Train Loss")
ax2.plot(history_dict["val_loss"], label="Val Loss")
ax2.set_title("Model Loss")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "training_curves.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Chunk-level evaluation
y_pred = model.predict(X_test, verbose=0)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

print("=" * 50)
print("CHUNK-LEVEL EVALUATION")
print("=" * 50)
print(classification_report(y_true_classes, y_pred_classes, target_names=GENRES))
print(f"Chunk-level accuracy: {accuracy_score(y_true_classes, y_pred_classes):.4f}")

In [ ]:
# File-level evaluation with majority voting
file_predictions = []
file_true_labels = []

for (genre, filename), group in test_file_map.groupby(["genre", "original_file"]):
    indices = group.index.values
    chunk_preds = y_pred_classes[indices]
    vote_counts = Counter(chunk_preds)
    majority_genre_idx = vote_counts.most_common(1)[0][0]
    file_predictions.append(majority_genre_idx)
    file_true_labels.append(GENRES.index(genre))

file_predictions = np.array(file_predictions)
file_true_labels = np.array(file_true_labels)

print("=" * 50)
print("FILE-LEVEL EVALUATION (MAJORITY VOTING)")
print("=" * 50)
print(classification_report(file_true_labels, file_predictions, target_names=GENRES))
print(f"File-level accuracy: {accuracy_score(file_true_labels, file_predictions):.4f}")

In [ ]:
# Confusion matrix
cm = confusion_matrix(file_true_labels, file_predictions)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=GENRES, yticklabels=GENRES)
plt.title("Confusion Matrix (File-Level, Majority Voting)")
plt.xlabel("Predicted Genre")
plt.ylabel("True Genre")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "confusion_matrix.png"), dpi=150, bbox_inches="tight")
plt.show()

## 8. Inference — Predict Genre of a Single File

Upload a `.wav` or `.mp3` file and predict its genre using majority voting across chunks.

**Tip:** Test with songs you find online — not from the training dataset! Try YouTube audio downloads or any music you have.

In [ ]:
def predict_genre(file_path, model):
    """
    Predict the genre of a .wav file using majority voting across chunks.

    Returns dict with predicted_genre, confidence, chunk_predictions, genre_probabilities.
    """
    X = preprocess_single_file(file_path)
    predictions = model.predict(X, verbose=0)

    chunk_pred_indices = np.argmax(predictions, axis=1)
    chunk_pred_genres = [GENRES[i] for i in chunk_pred_indices]

    vote_counts = Counter(chunk_pred_indices)
    winner_idx, winner_count = vote_counts.most_common(1)[0]
    confidence = winner_count / len(chunk_pred_indices)

    avg_probs = predictions.mean(axis=0)
    genre_probabilities = {genre: float(prob) for genre, prob in zip(GENRES, avg_probs)}

    return {
        "predicted_genre": GENRES[winner_idx],
        "confidence": confidence,
        "chunk_predictions": chunk_pred_genres,
        "genre_probabilities": genre_probabilities,
    }

In [ ]:
# Upload and predict (on Colab)
try:
    from google.colab import files
    print("Upload a .wav or .mp3 file to classify:")
    uploaded = files.upload()
    if uploaded:
        audio_path = list(uploaded.keys())[0]
        result = predict_genre(audio_path, model)
        print(f"\nPredicted Genre: {result['predicted_genre']}")
        print(f"Confidence: {result['confidence']:.1%}")
        print(f"\nChunk predictions: {result['chunk_predictions']}")
        print(f"\nGenre probabilities:")
        for genre, prob in sorted(result["genre_probabilities"].items(), key=lambda x: -x[1]):
            print(f"  {genre:12s}: {prob:.4f}")
except ImportError:
    # Not running on Colab — provide a file path manually
    # audio_path = "path/to/your/file.mp3"
    # result = predict_genre(audio_path, model)
    print("Not running on Colab. Set audio_path manually and call predict_genre(audio_path, model).")

## 9. Download Trained Model

Download the model and artifacts to your local machine.

In [ ]:
try:
    from google.colab import files
    files.download(MODEL_PATH)
    files.download(HISTORY_PATH)
    files.download(os.path.join(PLOTS_DIR, "training_curves.png"))
    files.download(os.path.join(PLOTS_DIR, "confusion_matrix.png"))
except ImportError:
    print(f"Model saved at: {MODEL_PATH}")
    print(f"History saved at: {HISTORY_PATH}")
    print(f"Plots saved in: {PLOTS_DIR}")